In [1]:
import ollama

In [2]:
[print(i) for i in ollama.list().models]

model='qwen3.5:4b' modified_at=datetime.datetime(2026, 11, 22, 14, 33, 41, 356506, tzinfo=TzInfo(10800)) digest='d8b0f5e9760cd1682034f292d7ef72ec46f432149be0df7574bf2d6e92e38c04' size=3324173934 details=ModelDetails(parent_model='model.gguf', format='gguf', family='qwen35', families=['qwen35'], parameter_size='4.2B', quantization_level='Q4_K_M')
model='llama3.1:8b' modified_at=datetime.datetime(2025, 7, 6, 13, 36, 38, 320743, tzinfo=TzInfo(10800)) digest='46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e' size=4920753328 details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='8.0B', quantization_level='Q4_K_M')


[None, None]

In [54]:
r_qwen = ollama.chat(model="qwen3:8b", messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [48]:
r = ollama.chat(model="llama3.2:3b", messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [108]:
r_qwen_mlx = ollama.chat(model="qwen3.5:9b-mlx", think=False, messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [3]:
r_qwen_mlx_4b = ollama.chat(model="qwen3.5:4b", think=False, messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [4]:
print(
      '\n \n QWEN 3.5 4B MLX: \n', r_qwen_mlx_4b.message.content
     )


 
 QWEN 3.5 4B MLX: 
 **Нет, это утверждение ложно.** Историческая хронология показывает, что эти两位 мыслителей не могли быть учителем и учеником друг друга.

Обратимся к датам их жизни:
*   **Платон** жил примерно с 428/427 до 348/347 гг. до н. э.
*   **Аристотель** родился около 384 г. до н. э. и умер в 322 г. до н. э.

Отсюда следует следующая последовательность событий:
1.  Платон учился под руководством Сократа (известного философа).
2.  Когда Аристотелю было около 17 лет (около 367 года до н.э.), он стал учеником Платона в Академии в Афинах на несколько лет. То есть сам **Аристотель был учеником Платона**.
3.  После смерти Платона Аристотель ушел из его круга и стал одним из крупнейших независимых философов той эпохи, а позже возглавил школу Ликеев в Стагиере, где воспитывал своих собственных учеников (например, Клеанта).

Таким образом, их отношение к вопросу было наоборот: **Аристотель был учеником Платона**, а не наоборот.


In [5]:
for response in [r_qwen_mlx_4b]:
    print("\n", response.model)
    print("Причина остановки:", response.done_reason)
    print("Входные токены:", response.prompt_eval_count)
    print("Выходные токены:", response.eval_count)
    print("Время генерации:", response.eval_duration / 1e9)
    print("Токенов/с:", response.eval_count * 1e9 / response.eval_duration)
    # print("Thinking:", len(response.message.thinking))
    # print("Content:", len(response.message.content))


 qwen3.5:4b
Причина остановки: stop
Входные токены: 32
Выходные токены: 306
Время генерации: 3.002659
Токенов/с: 101.90967405889246


In [75]:
from ollama import chat
from pathlib import Path
from pydantic import BaseModel, Field

use_model = "qwen3.5:9b"

text = Path("aristotle.txt").read_text(encoding="utf-8")

class Triple(BaseModel):
    subject: str
    predicate: str
    object: str


class Result(BaseModel):
    triples: list[Triple]


response_b_9b = chat(
    model = use_model,
    think = False,
    messages = [
        {
            "role": "system",
            "content": (
                "Извлекай из текста отношения в виде триплетов "
                "(субъект, предикат, объект). "
                "Предикат выражает отношение субъекта к объекту. "
                "Извлекай только явно выраженные отношения. "
                "Если отношение нельзя представить полноценным триплетом, пропусти его. "
                "Не добавляй информацию, отсутствующую в тексте. "
            )
        },
        {
            "role": "user",
            "content": text
        }
    ],
    format=Result.model_json_schema(),
    options={"temperature": 0},
)

In [65]:
result = Result.model_validate_json(response_b.message.content)

print("Токенов/с:", response_b.eval_count * 1e9 / response_b.eval_duration)
print("с:", response_b.eval_duration / 1e9)
print("\nТекст:\n")
print(text)
print("\nОтвет:\n")

for triple in result.triples:
    print(triple)

Токенов/с: 63.735433766644725
с: 6.69957

Текст:

Аристо́тель (др.-греч. Ἀριστοτέλης, 384 до н.э. — 322 до н.э.) — греческий философ и эрудит классического периода в Древней Греции, яркий пример «универсального человека».

Обученный Платоном, он был основателем перипатетической школы философии в Ликее и более широкой аристотелевской традиции. Его труды охватывают многие предметы, включая физику, биологию, зоологию, метафизику, логику, этику, эстетику, поэзию, театр, музыку, риторику, психологию, лингвистику, экономику, политику, метеорологию, геологию и государственное управление. Аристотель представил сложный синтез различных философий, существовавших до него. Прежде всего от его учений Запад унаследовал свой интеллектуальный лексикон, а также проблемы и методы исследования. В результате его философия оказала влияние почти на все формы знания на Западе и продолжает оставаться предметом современных философских дискуссий.

О его жизни известно немного. Аристотель родился в городе Стагир

- направленнность
- subject='Аристотель' predicate='попросил' object='Филиппа II Македонского обучать Александра Великого'
- слишком большие предикаты

### Два логических шага

In [67]:
MODEL = "qwen3.5:4b"

In [69]:
response_step_1 = chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Преобразуй текст в простые самостоятельные предложения. "
                "Каждое предложение должно содержать одно утверждение. "
                "Заменяй местоимения именами соответствующих сущностей. "
                "Сохраняй все факты, отрицания, неопределённость "
                "и временные связи. "
                "Не добавляй информацию от себя."
            )
        },
        {"role": "user", "content": text}
    ],
    think=False,
    options={"temperature": 0}
)

In [73]:
response_2steps = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Извлекай отношения из текста в виде триплетов "
                    "(субъект, предикат, объект). "
                    "Предикат должен выражать отношение субъекта к объекту. "
                    "Извлекай только явно выраженные отношения. "
                    "Не добавляй информацию от себя."
                )
            },
            {"role": "user", "content": response_step_1.message.content}
        ],
        format=Result.model_json_schema(),
        think=False,
        options={"temperature": 0}
    )

In [77]:
for response_name, response in {"Baseline 9B" : response_b,
                                #"Onthological" : response_o,
                                "Two-Steps 3B + 3B": response_2steps
                               }.items():
    print(f"\n=== Response: {response_name} ===\n")
    print("Токенов/с:", response.eval_count * 1e9 / response.eval_duration)
    print("с:", response.eval_duration / 1e9)
    print("\nТекст:\n")
    print(text)
    print("\nОтвет:\n")
    result = Result.model_validate_json(response.message.content)
    for triple in result.triples:
        print(triple)


=== Response: Baseline 9B ===

Токенов/с: 63.735433766644725
с: 6.69957

Текст:

Аристо́тель (др.-греч. Ἀριστοτέλης, 384 до н.э. — 322 до н.э.) — греческий философ и эрудит классического периода в Древней Греции, яркий пример «универсального человека».

Обученный Платоном, он был основателем перипатетической школы философии в Ликее и более широкой аристотелевской традиции. Его труды охватывают многие предметы, включая физику, биологию, зоологию, метафизику, логику, этику, эстетику, поэзию, театр, музыку, риторику, психологию, лингвистику, экономику, политику, метеорологию, геологию и государственное управление. Аристотель представил сложный синтез различных философий, существовавших до него. Прежде всего от его учений Запад унаследовал свой интеллектуальный лексикон, а также проблемы и методы исследования. В результате его философия оказала влияние почти на все формы знания на Западе и продолжает оставаться предметом современных философских дискуссий.

О его жизни известно немного. Ар

In [78]:
for response_name, response in {
    "Baseline 9B": response_b,
    "Two-Steps 3B + 3B": response_2steps
}.items():
    print(f"\n=== Response: {response_name} ===\n")

    input_tokens = response.prompt_eval_count
    output_tokens = response.eval_count

    print("Входных токенов:", input_tokens)
    print("Выходных токенов:", output_tokens)
    print("Всего токенов:", input_tokens + output_tokens)
    print("Токенов/с:", output_tokens * 1e9 / response.eval_duration)
    print("Генерация, с:", response.eval_duration / 1e9)
    print("Полное время, с:", response.total_duration / 1e9)

    print("\nТекст:\n", text)
    print("\nОтвет:\n")

    result = Result.model_validate_json(response.message.content)
    for triple in result.triples:
        print(triple)


=== Response: Baseline 9B ===

Входных токенов: 563
Выходных токенов: 427
Всего токенов: 990
Токенов/с: 63.735433766644725
Генерация, с: 6.69957
Полное время, с: 9.3562186

Текст:
 Аристо́тель (др.-греч. Ἀριστοτέλης, 384 до н.э. — 322 до н.э.) — греческий философ и эрудит классического периода в Древней Греции, яркий пример «универсального человека».

Обученный Платоном, он был основателем перипатетической школы философии в Ликее и более широкой аристотелевской традиции. Его труды охватывают многие предметы, включая физику, биологию, зоологию, метафизику, логику, этику, эстетику, поэзию, театр, музыку, риторику, психологию, лингвистику, экономику, политику, метеорологию, геологию и государственное управление. Аристотель представил сложный синтез различных философий, существовавших до него. Прежде всего от его учений Запад унаследовал свой интеллектуальный лексикон, а также проблемы и методы исследования. В результате его философия оказала влияние почти на все формы знания на Западе и 

#### ChatGPT eval

In [79]:
tokens_2steps = sum(
    r.prompt_eval_count + r.eval_count
    for r in (response_step_1, response_2steps)
)

print("Токенов на триплет:", tokens_2steps / 36)
print("Токенов на корректный триплет:", tokens_2steps / 33)

Токенов на триплет: 94.83333333333333
Токенов на корректный триплет: 103.45454545454545


| Метрика | Baseline 9B | Two-Steps 3B + 3B |
|---|---:|---:|
| Извлечено триплетов | 10 | 36 |
| Подтверждено текстом | 10 | 33 |
| Precision | 100% | 91,7% |
| Recall* | 21,7% | 71,7% |
| F1* | 35,7% | 80,5% |
| Токенов на триплет | 99 | 94,8 |
| Токенов на корректный триплет | 99 | 103,5 |


- Возможно оценка смещена из-за отношение с **труды охватывают** 

#### Plato retest

In [81]:
from ollama import chat
from pathlib import Path
from pydantic import BaseModel, Field

use_model = "qwen3.5:9b"

text = Path("plato.txt").read_text(encoding="utf-8")

class Triple(BaseModel):
    subject: str
    predicate: str
    object: str


class Result(BaseModel):
    triples: list[Triple]


response_b_9b_plato = chat(
    model = use_model,
    think = False,
    messages = [
        {
            "role": "system",
            "content": (
                "Извлекай из текста отношения в виде триплетов "
                "(субъект, предикат, объект). "
                "Предикат выражает отношение субъекта к объекту. "
                "Извлекай только явно выраженные отношения. "
                "Если отношение нельзя представить полноценным триплетом, пропусти его. "
                "Не добавляй информацию, отсутствующую в тексте. "
            )
        },
        {
            "role": "user",
            "content": text
        }
    ],
    format=Result.model_json_schema(),
    options={"temperature": 0},
)

In [83]:
def two_step(model):
    response_step_1_plato, response_2steps_plato = chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Преобразуй текст в простые самостоятельные предложения. "
                    "Каждое предложение должно содержать одно утверждение. "
                    "Заменяй местоимения именами соответствующих сущностей. "
                    "Сохраняй все факты, отрицания, неопределённость "
                    "и временные связи. "
                    "Не добавляй информацию от себя."
                )
            },
            {"role": "user", "content": text}
        ],
        think=False,
        options={"temperature": 0}
    )
    
    response_2steps_plato = chat(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Извлекай отношения из текста в виде триплетов "
                        "(субъект, предикат, объект). "
                        "Предикат должен выражать отношение субъекта к объекту. "
                        "Извлекай только явно выраженные отношения. "
                        "Не добавляй информацию от себя."
                    )
                },
                {"role": "user", "content": response_step_1_plato.message.content}
            ],
            format=Result.model_json_schema(),
            think=False,
            options={"temperature": 0}
    )

    return response_step_1_plato, response_2steps_plato

response_step_1_plato, response_2steps_plato = two_step(MODEL)

In [87]:
for response_name, response in {
    "Baseline 9B": response_b_9b_plato,
    "Two-Steps 3B + 3B": response_2steps_plato
}.items():
    print(f"\n=== Response: {response_name} ===\n")

    input_tokens = response.prompt_eval_count
    output_tokens = response.eval_count

    print("Входных токенов:", input_tokens)
    print("Выходных токенов:", output_tokens)
    print("Всего токенов:", input_tokens + output_tokens)
    print("Токенов/с:", output_tokens * 1e9 / response.eval_duration)
    print("Генерация, с:", response.eval_duration / 1e9)
    print("Полное время, с:", response.total_duration / 1e9)

    print("\nТекст:\n", text)
    print("\nОтвет:\n")

    result = Result.model_validate_json(response.message.content)
    for triple in result.triples:
        print(triple)


=== Response: Baseline 9B ===

Входных токенов: 468
Выходных токенов: 290
Всего токенов: 758
Токенов/с: 64.83318421700963
Генерация, с: 4.473018
Полное время, с: 7.5069996

Текст:
                 Плато́н (др.-греч. Πλάτων; 428/427 или 424/423 — 348/347 до н. э.), настоящее имя Аристо́кл — афинский философ классического периода Древней Греции, основатель платонической школы мысли и Академии, первого высшего учебного заведения в западном мире.
                Он считается одним из самых важных и влиятельных людей в истории человечества[1], и ключевой фигурой в истории древнегреческой и западной философии — вместе со своим учителем Сократом и самым известным из своих учеников Аристотелем[~ 1]. Часто Платона также называют одним из основателей западной религии и духовности[2]. Так называемый неоплатонизм философов, таких как Плотин и Порфирий, сильно повлиял на христианство через таких отцов церкви как Августин. Альфред Норт Уайтхед однажды заметил: «Наиболее правдоподобная общая характе